# 01 — Data Ingestion

Loads and consolidates the raw Sackmann ATP/WTA match history, ranking, and
player files, plus two "playing style" sources (Match Charting Project and
Grand Slam point-by-point data), into clean parquet files under
`data/processed/`.

**Data license**: all sources here are Jeff Sackmann / Tennis Abstract data,
released under CC BY-NC-SA 4.0 — attribution required, non-commercial use
only. See `data/external/*/UPSTREAM_README.md` for the original notices.


In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import data_loading as dl

pd.set_option('display.max_columns', 30)


## 1. Tour-level match history (ATP + WTA, 1968–2026)

In [2]:
atp_matches = dl.load_tour_matches('atp')
wta_matches = dl.load_tour_matches('wta')

matches = pd.concat([atp_matches, wta_matches], ignore_index=True)
print(f"ATP: {atp_matches.shape}, WTA: {wta_matches.shape}, combined: {matches.shape}")
matches[['tour', 'tourney_date']].groupby('tour').agg(['min', 'max'])


ATP: (199389, 50), WTA: (162182, 50), combined: (361571, 50)


tourney_date           
              min        max
tour                        
ATP    1967-12-28 2026-05-25
WTA    1967-12-25 2026-05-25

In [3]:
matches.to_parquet('../data/processed/matches_all.parquet', index=False)
print('saved matches_all.parquet:', matches.shape)


saved matches_all.parquet: (361571, 50)


### Sanity checks

Row counts by year (both tours should show a roughly stable ~2,500–3,000
matches/year at tour level in recent decades) and a null check on the columns
the modeling notebooks will depend on most.


In [4]:
matches['year'] = matches['tourney_date'].dt.year
display(matches.groupby(['year', 'tour']).size().unstack().tail(10))

key_cols = ['winner_rank', 'loser_rank', 'surface', 'best_of', 'round', 'score']
print(matches[key_cols].isna().mean().rename('null_fraction'))


tour,ATP,WTA
year,,
2017,2911,2862
2018,2982,2847
2019,2721,2652
2020,1462,1276
2021,2733,2597
2022,2917,2594
2023,2986,2810
2024,3158,2792
2025,2862,2692


winner_rank    0.241803
loser_rank     0.275365
surface        0.021813
best_of        0.000000
round          0.000000
score          0.000028
Name: null_fraction, dtype: float64


**Note on Slam filtering**: `tourney_level == 'G'` isolates Grand Slam
matches for the winner/margin modeling target. Full match history (all
levels) is kept here because Elo and form features need continuity across a
player's whole career, not just their Slam matches.


In [5]:
slam_matches = matches[matches['tourney_level'] == 'G']
print(slam_matches.shape)
slam_matches['tourney_name'].value_counts()


(54306, 51)


tourney_name
Roland Garros               14066
Wimbledon                   13998
US Open                     12382
Australian Open             11993
Us Open                      1651
Australian Open-2              63
Australian Chps.               61
Australian Championships       61
Australian Open 2              31
Name: count, dtype: int64

## 2. Players and rankings

In [6]:
atp_players = dl.load_players('atp')
wta_players = dl.load_players('wta')
players = pd.concat([atp_players, wta_players], ignore_index=True)
players.to_parquet('../data/processed/players.parquet', index=False)
print(players.shape)
players.head(3)


(137483, 9)


,player_id,name_first,name_last,hand,dob,ioc,height,wikidata_id,tour
0,100001,Gardnar,Mulloy,R,1913-11-22,USA,185.0,Q54544,ATP
1,100002,Pancho,Segura,R,1921-06-20,ECU,168.0,Q54581,ATP
2,100003,Frank,Sedgman,R,1927-10-02,AUS,180.0,Q962049,ATP


In [7]:
atp_rankings = dl.load_rankings('atp')
wta_rankings = dl.load_rankings('wta')
rankings = pd.concat([atp_rankings, wta_rankings], ignore_index=True)
rankings.to_parquet('../data/processed/rankings.parquet', index=False)
print(rankings.shape)
rankings.head(3)


(5573852, 6)


,ranking_date,rank,player_id,points,tour,tours
0,2000-01-10,1,101736,4135.0,ATP,NaN
1,2000-01-10,2,102338,2915.0,ATP,NaN
2,2000-01-10,3,101948,2419.0,ATP,NaN


## 3. Playing-style features — Match Charting Project

Forehand/backhand reliance, rally-length preference, and net game, aggregated
per player across every charted match (ATP + WTA). Coverage is volunteer-
selected — mostly top players and marquee matches, not a full census — so
expect this to be missing for many lower-ranked players.


In [8]:
def build_style_table(gender):
    shot = dl.build_shot_style_features(gender)
    rally = dl.build_rally_style_features(gender)
    net = dl.build_net_game_features(gender)

    out = shot.merge(rally.drop(columns=['tour']), on='player', how='outer')
    out = out.merge(net.drop(columns=['tour']), on='player', how='outer')
    return out

style_atp = build_style_table('m')
style_wta = build_style_table('w')
print('ATP charted players:', style_atp.shape[0], '| WTA charted players:', style_wta.shape[0])
style_atp.head(3)


ATP charted players: 1002 | WTA charted players: 730


,player,serve_pts,return_pts,winners,winners_fh,winners_bh,unforced,unforced_fh,unforced_bh,fh_winner_share,bh_winner_share,fh_unforced_share,bh_unforced_share,tour,pts_long_rally,pts_medium_rally,pts_short_rally,pts_very_long_rally,pts_long_rally_share,pts_medium_rally_share,pts_short_rally_share,pts_very_long_rally_share,net_point_win_rate,snv_rate
0,Aaron Krickstein,816,765,191,76,76,214,85,118,0.500000,0.500000,0.418719,0.581281,ATP,178.0,399.0,778.0,226.0,0.112587,0.252372,0.492094,0.142948,0.591304,0.056995
1,Abedallah Shelbayh,150,146,52,31,9,59,36,20,0.775000,0.225000,0.642857,0.357143,ATP,16.0,65.0,188.0,27.0,0.054054,0.219595,0.635135,0.091216,0.723404,0.068027
2,Adam Pavlasek,188,176,53,22,9,91,39,47,0.709677,0.290323,0.453488,0.546512,ATP,35.0,85.0,216.0,28.0,0.096154,0.233516,0.593407,0.076923,0.649123,0.063830


## 4. Playing-style features — Grand Slam point-by-point (serve speed, FH/BH winners)

Serve speed and forehand-vs-backhand winner tendency from the four majors'
own ball-tracking data (2011–2024). This is the only public source of actual
serve-speed numbers; there is no public groundstroke-speed data.

Caveats (see `data/external/tennis-sackmann-archive/slam_pointbypoint/UPSTREAM_README.md`):
column availability varies by slam/year across two different data providers,
the Australian Open and French Open stopped being scraped after 2022, and
player names are matched by string (not player_id), so a handful of players
with inconsistent name formatting across years will be undercounted.


In [9]:
slam_style = dl.build_slam_style_features()
print(slam_style.shape)
slam_style['tour'].value_counts(dropna=False)


(1680, 7)


tour
ATP    851
WTA    829
Name: count, dtype: int64

In [10]:
slam_style_atp = slam_style[slam_style['tour'] == 'ATP'].drop(columns=['tour'])
slam_style_wta = slam_style[slam_style['tour'] == 'WTA'].drop(columns=['tour'])

style_atp_full = style_atp.merge(slam_style_atp, on='player', how='outer')
style_wta_full = style_wta.merge(slam_style_wta, on='player', how='outer')

style_atp_full.to_parquet('../data/processed/style_features_atp.parquet', index=False)
style_wta_full.to_parquet('../data/processed/style_features_wta.parquet', index=False)
print('ATP style rows:', style_atp_full.shape, '| WTA style rows:', style_wta_full.shape)


ATP style rows: (1424, 29) | WTA style rows: (1192, 29)


## 5. Ingestion summary


In [11]:
import os

print('data/processed/ contents:')
for f in sorted(os.listdir('../data/processed')):
    path = os.path.join('../data/processed', f)
    print(f'  {f:35s} {os.path.getsize(path) / 1e6:8.1f} MB')


data/processed/ contents:
  matches_all.parquet                     10.5 MB
  players.parquet                          2.6 MB
  rankings.parquet                        15.8 MB
  style_features_atp.parquet               0.2 MB
  style_features_wta.parquet               0.1 MB
